In [1]:
import os
from pyspark.sql import SparkSession

#Macbook Intel - Para rodar o Java precisei apontar esse caminho
os.environ["JAVA_HOME"] = "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

spark = SparkSession.builder \
    .appName("T032-LinearRegression") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "12g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("Motor do Spark versão:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 10:12:37 WARN Utils: Your hostname, MacBook-Pro-de-Gabriel.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.18 instead (on interface en0)
26/05/03 10:12:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 10:12:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Motor do Spark versão: 4.1.1


In [2]:
# Imports
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Inicializa a sessão do Spark
spark = SparkSession.builder \
    .appName("T032-LinearRegression") \
    .getOrCreate()

print("Spark rodando! Versão:", spark.version)

Spark rodando! Versão: 4.1.1


In [3]:
#Carregar os Dados
caminho_dados = "../data/Indian_Weather_Dataset.parquet" 

# Carrega o dataset
df = spark.read.parquet(caminho_dados)
print(f"Total de registros: {df.count()}")

# Divide os dados em Treino e Teste
df_treino, df_teste = df.randomSplit([0.7, 0.3], seed=42)
print(f"Registros de Treino: {df_treino.count()}")
print(f"Registros de Teste: {df_teste.count()}")

Total de registros: 46082160


Registros de Treino: 32258565


Registros de Teste: 13823595


In [5]:
colunas_features = [
    "humidity_pct", "pressure_hPa", "dew_point_C", 
    "solar_radiation_Wm2", "cloud_cover_pct", "wind_speed_ms", 
    "wind_dir_sin", "wind_dir_cos", "cape", "et0_mm", "precip_mm"
]

#Agrupar as features
assembler = VectorAssembler(inputCols=colunas_features, outputCol="raw_features")

#StandardScaller
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", 
                      withStd=True, withMean=True)

#Linear_regression
lr = LinearRegression(featuresCol="scaled_features", labelCol="temperature_C", 
                      regParam=0.1, elasticNetParam=0.0, solver="auto")

#Pipeline
pipeline = Pipeline(stages=[assembler, scaler, lr])
print("Pipeline configurada com sucesso")

Pipeline configurada com sucesso


In [ ]:
print("Treino")

# Treina o modelo
modelo_treinado = pipeline.fit(df_treino)

# Aplica o modelo nos dados
previsoes = modelo_treinado.transform(df_teste)
evaluator_rmse = RegressionEvaluator(labelCol="temperature_C", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="temperature_C", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="temperature_C", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(previsoes)
mae = evaluator_mae.evaluate(previsoes)
r2 = evaluator_r2.evaluate(previsoes)

print(f"\n--- RESULTADOS DO MODELO ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

Treino


26/05/03 10:16:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/03 10:17:06 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/03 10:18:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK



--- RESULTADOS DO MODELO ---
RMSE: 1.6437
MAE:  1.1375


In [ ]:
# Extrai o modelo do Pipeline
lr_model = modelo_treinado.stages[-1]

print("Intercepto (Temperatura Base):", round(lr_model.intercept, 4))
print("\nCoeficientes gerados pela Regularização Ridge:")

for feature, coef in zip(colunas_features, lr_model.coefficients):
    print(f"- {feature}: {round(coef, 4)}")

Intercepto (Temperatura Base): 23.6117

Coeficientes (Pesos) gerados pela Regularização Ridge:
- humidity_pct: -5.6666
- pressure_hPa: 0.3991
- dew_point_C: 6.7974
- solar_radiation_Wm2: -2.7982
- cloud_cover_pct: 0.259
- wind_speed_ms: -0.2248
- wind_dir_sin: -0.1113
- wind_dir_cos: -0.077
- cape: 0.0
- et0_mm: 3.6052
- precip_mm: 0.1435
